<a href="https://colab.research.google.com/github/Du-nara/ME421-Mechanical-Systems-Lab-A3/blob/main/Controls/E_20_256_controls.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Task 1 – Dynamic Model of the Twin Rotor System**

* Complete the activities in groups that were assigned for ME421 for the vibrations lab.

* Make a copy of this and save it in your group github group repository with your index number as the file name

* Do all your work, EXCLUSIVELY, in that saved notebook. Your github commits will serve as a reflection of your individual contributions.

# References

* https://github.com/mugalan/intrinsic-rigid-body-control-estimation/tree/main/rigid-body-control

## Setup: Install & Import Libraries

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp
from scipy.linalg import expm
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

Libraries loaded successfully.


# Task #1

### Derivation

**System Description:**
The twin rotor system consists of a rigid body (rotor assembly) that can rotate in 3D space. Two rotors generate forces:
- Rotor 1 (pitch): thrust along $\mathbf{b}_1$, moment about $\mathbf{b}_3$ axis, with angle $\alpha$
- Rotor 2 (yaw): thrust along $\mathbf{b}_1$, moment about $\mathbf{b}_3$ axis, with angle $\beta$

The rotation matrix $R \in SO(3)$ maps body-frame vectors to spatial (inertial) frame.

**Newton-Euler equations on $SO(3)$:**

The kinematics of the rotation matrix are:
$$\dot{R} = \hat{\omega} R$$
where $\hat{\omega}$ is the skew-symmetric matrix of the spatial angular velocity $\omega$.

The spatial angular momentum is:
$$\pi = R \mathbb{I} R^T \omega = R \mathbb{I} \Omega$$
where $\Omega = R^T \omega$ is the body-frame angular velocity, and $\mathbb{I}$ is the body-frame inertia tensor.

**Momentum dynamics:**

Differentiating $\pi$:
$$\dot{\pi} = \frac{d}{dt}(R \mathbb{I} \Omega) = \dot{R}\mathbb{I}\Omega + R\mathbb{I}\dot{\Omega}$$

By Euler's equation in the body frame:
$$\mathbb{I}\dot{\Omega} = -\Omega \times \mathbb{I}\Omega + \tau^{\text{body}}$$

So the spatial momentum equation becomes:
$$\dot{\pi} = \tau^e + \tau^u$$

with the recovery: $\omega = (\mathbb{I}^R)^{-1}\pi$ where $\mathbb{I}^R = R\mathbb{I}R^T$ is the spatially-resolved inertia.

**Control torque:**
$$\tau^u = R \begin{bmatrix}1 & 0\\ 0 & 0\\ 0 & 1\end{bmatrix} \begin{bmatrix}\cos\alpha & -\cos\beta\\ \sin\alpha & -\sin\beta\end{bmatrix} \begin{bmatrix}u_1\\ u_2\end{bmatrix}$$

This maps the two rotor thrust inputs $u_1, u_2$ to torques in the $\mathbf{b}_1$ and $\mathbf{b}_3$ directions.

**Constraint torque** (prevents rotation about $\mathbf{b}_2$):
$$\tau^e = R\begin{bmatrix}0\\T_2\\0\end{bmatrix}$$

The constraint force $T_2$ is determined by the constraint that the $\mathbf{b}_2$ component of angular acceleration is zero:
$$T_2 = e_2^T\left(\Omega \times \mathbb{I}\Omega + \mathbb{I}\dot{\Omega}\right)$$

This is solved implicitly from the equations of motion, ensuring the system respects its mechanical constraint.

**Summary — the twin rotor model:**
$$\boxed{\dot{R} = \hat{\omega}R, \qquad \dot{\pi} = \tau^e + \tau^u, \qquad \omega = (\mathbb{I}^R)^{-1}\pi}$$

## Helper Functions: SO(3) Utilities

In [2]:
# ============================================================
# SO(3) UTILITY FUNCTIONS
# ============================================================

def hat(v):
    """Skew-symmetric matrix (hat map) for vector v in R^3."""
    v = np.asarray(v).flatten()
    return np.array([
        [ 0,    -v[2],  v[1]],
        [ v[2],  0,    -v[0]],
        [-v[1],  v[0],  0   ]
    ])

def vee(S):
    """Inverse of hat map: extract vector from skew-symmetric matrix."""
    return np.array([S[2,1], S[0,2], S[1,0]])

def project_SO3(R):
    """Project matrix onto SO(3) using SVD to avoid numerical drift."""
    U, _, Vt = np.linalg.svd(R)
    D = np.diag([1, 1, np.linalg.det(U @ Vt)])
    return U @ D @ Vt

def cross(a, b):
    """Cross product a × b."""
    return np.cross(a, b)

def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1,0,0],[0,c,-s],[0,s,c]])

def Ry(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,0,s],[0,1,0],[-s,0,c]])

def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

print('SO(3) utility functions defined.')

SO(3) utility functions defined.


## System Parameters

In [ ]:
# ============================================================
# TWIN ROTOR SYSTEM PARAMETERS
# ============================================================

# Inertia tensor (body frame) — diagonal for a symmetric rotor assembly
# Units: kg·m²
I_xx = 0.05  # Pitch axis inertia
I_yy = 0.10  # (constrained) axis inertia
I_zz = 0.05  # Yaw axis inertia
I_body = np.diag([I_xx, I_yy, I_zz])
I_body_inv = np.linalg.inv(I_body)

# Rotor geometry angles (radians)
# alpha: pitch rotor angle from b1 axis
# beta:  yaw rotor angle from b1 axis
alpha = 0.0          # pitch rotor aligned with b1
beta  = np.pi / 2.0  # yaw rotor at 90 degrees

# Control input mapping matrix B (3x2)
# tau^u = R @ B_mat @ [u1; u2]
B_geom = np.array([[1, 0],
                   [0, 0],
                   [0, 1]])  # selects b1 and b3 axes

B_rot = np.array([[np.cos(alpha), -np.cos(beta)],
                  [np.sin(alpha), -np.sin(beta)]])

B_mat = B_geom @ B_rot  # 3x2 control input matrix

print('System parameters set.')
print(f'Inertia tensor:\n{I_body}')
print(f'Control input matrix B:\n{B_mat}')

System parameters set.
Inertia tensor:
[[0.05 0.   0.  ]
 [0.   0.1  0.  ]
 [0.   0.   0.05]]
Control input matrix B:
[[ 1.000000e+00 -6.123234e-17]
 [ 0.000000e+00  0.000000e+00]
 [ 0.000000e+00 -1.000000e+00]]


## Equations of Motion

In [3]:
# ============================================================
# EQUATIONS OF MOTION FOR TWIN ROTOR SYSTEM
# ============================================================

def compute_constraint_torque(R, Omega, u1, u2):
    """
    Compute the constraint torque T2 that enforces zero angular
    acceleration about the b2 axis.

    From the equation of motion in body frame:
        I * dOmega/dt = -Omega × I*Omega + tau_body

    The b2 component of tau_body comes only from T2.
    Control torque has NO b2 component (by design of B_mat).

    For constraint: e2^T (Omega × I*Omega + I*dOmega/dt) is solved
    with dOmega_2/dt = 0 (constraint).

    This gives:
        T2 = e2^T (Omega × I*Omega) - e2^T (I * (dOmega_1/dt * e1 + dOmega_3/dt * e3))

    However, for simulation we treat the system as if the constraint
    simply removes b2 degree of freedom and T2 maintains it.
    We compute T2 = e2^T(Omega × I*Omega + I*dOmega/dt) evaluated
    at dOmega_2 = 0.
    """
    IOmega = I_body @ Omega
    gyro   = cross(Omega, IOmega)    # Omega × I*Omega

    # Control torque in body frame
    u_vec = np.array([u1, u2])
    tau_u_body = B_mat @ u_vec       # 3-vector in body frame

    # For the unconstrained b1, b3 components:
    # I * dOmega/dt = -gyro + tau_u_body + [0, T2, 0]^T
    # For b2: I_yy * 0 = -gyro[1] + 0 + T2  =>  T2 = gyro[1]
    T2 = gyro[1]
    return T2


def twin_rotor_ode(t, state, u_func):
    """
    ODE for the twin rotor system.

    State vector (12 components):
        state[0:9]  = R flattened (row-major)
        state[9:12] = pi (spatial angular momentum)

    Returns d/dt(state).
    """
    R   = state[0:9].reshape(3, 3)
    pi  = state[9:12]

    # Spatially-resolved inertia
    IR  = R @ I_body @ R.T
    IR_inv = np.linalg.inv(IR)

    # Angular velocity
    omega = IR_inv @ pi

    # Body-frame angular velocity
    Omega = R.T @ omega

    # Control inputs
    u1, u2 = u_func(t, R, pi, omega)

    # Control torque in spatial frame
    u_vec   = np.array([u1, u2])
    tau_u   = R @ B_mat @ u_vec

    # Constraint torque (maintains b2 constraint)
    T2 = compute_constraint_torque(R, Omega, u1, u2)
    tau_e = R @ np.array([0.0, T2, 0.0])

    # Kinematics: dR/dt = hat(omega) @ R
    dR = hat(omega) @ R

    # Momentum dynamics: dpi/dt = tau_e + tau_u
    dpi = tau_e + tau_u

    d_state = np.concatenate([dR.flatten(), dpi])
    return d_state


def simulate(u_func, t_span, t_eval, R0=None, pi0=None):
    """
    Simulate the twin rotor system.

    Args:
        u_func:  callable u_func(t, R, pi, omega) -> (u1, u2)
        t_span:  (t0, tf)
        t_eval:  array of times at which to record state
        R0:      initial rotation matrix (default: identity)
        pi0:     initial spatial momentum (default: zero)

    Returns:
        t_eval, R_traj (N,3,3), omega_traj (N,3), pi_traj (N,3)
    """
    if R0 is None:
        R0 = np.eye(3)
    if pi0 is None:
        pi0 = np.zeros(3)

    state0 = np.concatenate([R0.flatten(), pi0])

    sol = solve_ivp(
        fun=lambda t, s: twin_rotor_ode(t, s, u_func),
        t_span=t_span,
        y0=state0,
        t_eval=t_eval,
        method='RK45',
        rtol=1e-8,
        atol=1e-10
    )

    N = len(sol.t)
    R_traj    = np.zeros((N, 3, 3))
    pi_traj   = np.zeros((N, 3))
    omega_traj = np.zeros((N, 3))

    for i in range(N):
        R_i  = sol.y[0:9, i].reshape(3, 3)
        R_i  = project_SO3(R_i)  # keep on SO(3)
        pi_i = sol.y[9:12, i]
        IR_i = R_i @ I_body @ R_i.T
        omega_i = np.linalg.inv(IR_i) @ pi_i

        R_traj[i]    = R_i
        pi_traj[i]   = pi_i
        omega_traj[i] = omega_i

    return sol.t, R_traj, omega_traj, pi_traj


print('Equations of motion defined.')

Equations of motion defined.
